# 02 - BERT-based sentiment labelling (GPU)

Three independent approaches produce a label (negative / neutral / positive) and an integer 1–10 score from the review text.

| | Approach | Training data | Training |
|---|---|---|---|
| **A** | Zero-shot | none (ready models) | none |
| **B** | Transfer learning | HRAST (human labels) | BERT fine-tuning |
| **C** | Self-training | Booking reviews with the most confident labels of A and B | BERT fine-tuning |

Runs on Google Colab (T4 GPU) with the `CODE` folder on Google Drive, or on a 4 GB GPU with a smaller batch (`config.py`). Predictions are saved in chunks, so an interrupted run continues where it stopped.

| Step | T4, 20k reviews (estimate) |
|---|---|
| A (2 models) | 3–5 min |
| B training + labelling | 15 + 5 min |
| C training + labelling | 15 + 2 min |

## 1. Environment

In [1]:
# On Colab: mount Drive, change to the project folder, install packages.
# PROJECT_DIR is the project folder on Google Drive.
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/THESIS/CODE"
    os.chdir(os.path.join(PROJECT_DIR, "notebooks"))
    get_ipython().system("pip install -q transformers accelerate pyarrow")
print("Working directory:", os.getcwd())

Working directory: c:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\notebooks


In [2]:
# The notebooks run from notebooks/, so the project root is added to the path.
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import set_seed
from src.config import CFG, PATHS, SENTIMENT_COLORS, INK, APPROACHES, apply_plot_style
from src import data_loading as dl
from src import preprocessing as pp
from src import evaluation as ev
from src import bert_labelling as bl
from src import bert_training as bt
from src import label_validation as lv
from src import pipeline

set_seed()
PATHS.ensure()
apply_plot_style()
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 170)

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("!!! NO GPU — labelling will be very slow on a CPU !!!")

PyTorch 2.14.0+cu132 | CUDA: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU | VRAM: 4.3 GB


In [3]:
working = dl.read_frame(pipeline.WORKING_SET)
print(f"Working set: {len(working):,} reviews")

Working set: 203,328 reviews


In [4]:
# Real token length with the 3-class model's tokenizer (on a sample of 2000).
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(CFG.zero_shot_3class_model)
sample_texts = working["text_bert"].sample(min(2000, len(working)), random_state=CFG.random_seed)
lens = np.array([len(tok(t)["input_ids"]) for t in sample_texts])
print(f"Tokens — median {np.median(lens):.0f}, 95%: {np.percentile(lens, 95):.0f}, max {lens.max()}")
print(f"{(lens > CFG.max_length).mean():.2%} of the reviews exceed max_length={CFG.max_length}.")

17:26:36 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
17:26:36 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
17:26:36 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cardiffnlp/twitter-roberta-base-sentiment-latest/3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7/config.json?%2Fcardiffnlp%2Ftwitter-roberta-base-sentiment-latest%2Fresolve%2Fmain%2Fconfig.json=&etag=%22baae943f01ad3371bc9d543de3229512dfe27cb8%22 "HTTP/1.1 200 OK"
17:26:36 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
17:26:36 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-

Tokens — median 47, 95%: 156, max 582
1.00% of the reviews exceed max_length=256.


## 2. (A) Zero-shot labelling

Two models fine-tuned by others, used without training:
* a 3-class model (`CFG.zero_shot_3class_model`) -> `A_label`;
* a 1–5 star model (`CFG.zero_shot_stars_model`) -> `A_score10`, from the expected number of stars $E=\sum_k k\,P(k)$ as $\text{score}=1+9\,(E-1)/4$, rounded.

Neither model was trained on hotel reviews (tweets, product reviews), so there is a domain gap.

In [5]:
scorers_a = {"3class": bl.TransformerScorer(CFG.zero_shot_3class_model),
             "stars": bl.TransformerScorer(CFG.zero_shot_stars_model)}
labels_a = bl.label_approach_a(working, scorers=scorers_a)
bl.save_labels(labels_a, "labels_A")

display(labels_a["A_label"].value_counts(normalize=True).round(3).to_frame("share"))
display(labels_a[["A_score10", "A_score10_polarity", "A_expected_stars"]].describe().round(2))

17:26:37 | INFO    | src.bert_labelling | Loading model: cardiffnlp/twitter-roberta-base-sentiment-latest (device: cuda, fp16: True)
17:26:37 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
17:26:37 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cardiffnlp/twitter-roberta-base-sentiment-latest/3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7/config.json?%2Fcardiffnlp%2Ftwitter-roberta-base-sentiment-latest%2Fresolve%2Fmain%2Fconfig.json=&etag=%22baae943f01ad3371bc9d543de3229512dfe27cb8%22 "HTTP/1.1 200 OK"
17:26:37 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
17:26:37 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/tokenizer_config.json

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
17:26:39 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-roberta-base-sentiment-latest/commits/main "HTTP/1.1 200 OK"
17:26:40 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-roberta-base-sentiment-latest/discussions?p=0 "HTTP/1.1 200 OK"
17:26:40 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-roberta-base-sentiment-latest/commits/refs%2Fpr%2F43 "HTTP/1.1 200 OK"
17:26:40 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnl

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

17:26:43 | INFO    | src.bert_labelling | Model type: stars | labels: [1, 2, 3, 4, 5]
17:26:43 | WARNING | src.bert_labelling | Cache 'A_3class_review' is stale (input or model changed) — recomputing.
17:26:43 | INFO    | src.bert_labelling | [A_3class_review] chunk 1/21 (10000 texts)
17:26:46 | INFO    | src.bert_labelling |   twitter-roberta-base-sentiment-latest: 50/157 batches (1022 texts/s)
17:26:50 | INFO    | src.bert_labelling |   twitter-roberta-base-sentiment-latest: 100/157 batches (869 texts/s)
17:26:58 | INFO    | src.bert_labelling |   twitter-roberta-base-sentiment-latest: 150/157 batches (622 texts/s)
17:27:00 | INFO    | src.bert_labelling | [A_3class_review] chunk 2/21 (10000 texts)
17:27:03 | INFO    | src.bert_labelling |   twitter-roberta-base-sentiment-latest: 50/157 batches (1274 texts/s)
17:27:07 | INFO    | src.bert_labelling |   twitter-roberta-base-sentiment-latest: 100/157 batches (897 texts/s)
17:27:16 | INFO    | src.bert_labelling |   twitter-roberta-base

,share
A_label,
positive,0.928
neutral,0.038
negative,0.034


,A_score10,A_score10_polarity,A_expected_stars
count,203328.00,203328.00,203328.00
mean,8.18,9.35,4.19
std,1.53,1.59,0.66
min,1.00,1.00,1.05
25%,8.00,9.00,3.89
50%,9.00,10.00,4.35
75%,9.00,10.00,4.69
max,10.00,10.00,4.99


In [6]:
# Internal agreement of the 3-class and the star model.
tab = lv.approach_crosstab(labels_a, "A_label", "A_star_label")
display(tab)
print(f"Cohen's kappa (3-class vs stars): {lv.cohen_kappa(labels_a['A_label'], labels_a['A_star_label']):.3f}")
ev.save_table(tab, "02_A_internal_agreement")

A_star_label,negative,neutral,positive
A_label,,,
negative,0.4382,0.4785,0.0833
neutral,0.1143,0.4331,0.4526
positive,0.0092,0.0899,0.9009


17:39:03 | INFO    | src.evaluation | Table saved: 02_A_internal_agreement.csv


Cohen's kappa (3-class vs stars): 0.338


WindowsPath('C:/Users/seszt/Documents/BCE/7_felev/Szakdolgozat/THESIS/CODE/results/tables/02_A_internal_agreement.csv')

## 3. (B) Transfer learning on HRAST

HRAST contains hand-annotated hotel review sentences. `bert-base-uncased` is fine-tuned on it, the Booking reviews are labelled sentence by sentence, and the sentence probabilities are averaged per review. Apart from the gold sample, the HRAST test set is the only evaluation on human labels. The HRAST csv is expected in `data/raw/hrast/`; HRAST is used only as training data of this labeller.

In [7]:
HRAST_AVAILABLE = dl.find_hrast_file() is not None
print("HRAST available:", HRAST_AVAILABLE)
if HRAST_AVAILABLE:
    hrast_raw = dl.load_hrast_raw()
    display(hrast_raw.head())
    # If columns are not recognised: prepare_hrast_sentences(hrast_raw, text_col="...", sentiment_col="...")
    hrast = dl.prepare_hrast_sentences(hrast_raw)
    display(hrast["label"].value_counts().to_frame("sentences"))

17:39:03 | INFO    | src.data_loading | HRAST loaded: hrast.csv — 23113 rows, columns: ['id', 'review', 'Unnamed: 2', 'positive', 'negative', 'neutral', 'Clean', 'Comfort', 'Facilities/Amenities', 'Location', 'Restaurant (dinner)', 'Staff', 'View (Balcony)', 'Breakfast', 'Room', 'Pool', 'Beach', 'Bathroom/Shower (toilet)', 'Bar', 'Bed', 'Parking', 'Noise', 'Reception-checkin', 'Lift', 'Value for money', 'Wi-Fi', 'Generic', 'Aspect']


HRAST available: True


,id,review,Unnamed: 2,positive,negative,neutral,Clean,Comfort,Facilities/Amenities,Location,...,Bar,Bed,Parking,Noise,Reception-checkin,Lift,Value for money,Wi-Fi,Generic,Aspect
0,1,1 elevator not working.,NaN,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,2,1 euros to park per day on their own premises.,NaN,0,0,1,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
2,3,1 hour queue to check-in.,NaN,0,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,4,1 min walk to metro entrance and also few min walk to the central station where you can access rail trains.,NaN,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,5,"1 minute walk from the main metro & train station, which made getting around Copenhagen extremely easy.",NaN,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


17:39:03 | INFO    | src.data_loading | HRAST columns used — text: review | sentiment: one-hot ['negative', 'neutral', 'positive'] | group: None
17:39:03 | INFO    | src.data_loading | HRAST one-hot rows — positive: 11818 | negative: 10503 | neutral only: 791 | both pos+neg: 1 | none: 0
17:39:05 | INFO    | src.data_loading | HRAST sentences: 23104 (dropped as conflicting: 1) | distribution: {'positive': 11812, 'negative': 10501, 'neutral': 791}


,sentences
label,
positive,11812
negative,10501
neutral,791


In [8]:
# Model (B) learns only from HRAST and does not depend on the Booking data, so an existing model
# is reused (also in the full run). RETRAIN_B = True retrains it.
RETRAIN_B = False
model_b_ready = (PATHS.models / "approach_b_hrast" / "config.json").exists()
if HRAST_AVAILABLE and (RETRAIN_B or not model_b_ready):
    result_b = bt.train_approach_b(hrast)
    hrast_test = pd.DataFrame([result_b["test_metrics"]], index=["HRAST test"]).round(4)
    display(hrast_test)
    ev.save_table(hrast_test, "02_approach_b_hrast_test")
    ev.plot_confusion_matrix(result_b["test_predictions"]["label"], result_b["test_predictions"]["predicted"],
                             title="(B) on HRAST test", save_as="02_approach_b_hrast_confusion")
    plt.show()
elif model_b_ready:
    print("(B) is already trained on HRAST — reused, not retrained.")
    if (PATHS.tables / "02_approach_b_hrast_test.csv").exists():
        display(pd.read_csv(PATHS.tables / "02_approach_b_hrast_test.csv", index_col=0))

(B) is already trained on HRAST — reused, not retrained.


,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
HRAST test,0.9414,0.7731,0.7825,0.7776,0.9431,0.9414,0.9422


In [9]:
if HRAST_AVAILABLE:
    scorer_b = bl.TransformerScorer(str(PATHS.models / "approach_b_hrast"))
    labels_b = bl.label_approach_b(working, scorer=scorer_b)
    bl.save_labels(labels_b, "labels_B")
    display(labels_b["B_label"].value_counts(normalize=True).round(3).to_frame("share"))
    print(f"Mixed reviews (a confidently positive AND negative sentence): {labels_b['B_is_mixed'].mean():.1%}")

    # Keep sentence-level predictions for the aspect analysis (no recomputation thanks to the cache).
    sentences_b = bl.predict_sentences(working, "", "B", cache_name="B_sentence", scorer=scorer_b)
    dl.write_frame(sentences_b, PATHS.labels / "sentences_B")

17:39:05 | INFO    | src.bert_labelling | Loading model: C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\results\models\approach_b_hrast (device: cuda, fp16: True)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

17:39:06 | INFO    | src.bert_labelling | Model type: 3class | labels: ['negative', 'neutral', 'positive']
17:39:10 | INFO    | src.bert_labelling | 203328 reviews -> 917580 sentences
17:39:10 | WARNING | src.bert_labelling | Cache 'B_sentence_sent' is stale (input or model changed) — recomputing.
17:39:10 | INFO    | src.bert_labelling | [B_sentence_sent] chunk 1/92 (10000 texts)
17:39:12 | INFO    | src.bert_labelling |   C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\results\models\approach_b_hrast: 50/157 batches (2242 texts/s)
17:39:13 | INFO    | src.bert_labelling |   C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\results\models\approach_b_hrast: 100/157 batches (2082 texts/s)
17:39:16 | INFO    | src.bert_labelling |   C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\results\models\approach_b_hrast: 150/157 batches (1760 texts/s)
17:39:16 | INFO    | src.bert_labelling | [B_sentence_sent] chunk 2/92 (10000 texts)
17:39:18 | INFO    | s

,share
B_label,
positive,0.898
negative,0.098
neutral,0.004


Mixed reviews (a confidently positive AND negative sentence): 44.9%


## 4. (C) Self-training on the Booking data

The most confident labels on which the teachers (A) and (B) agree form the training set (`CFG.pseudo_label_min_confidence`, `CFG.pseudo_label_require_agreement`). The BERT model fine-tuned on them learns from Booking text and also labels the uncertain reviews. Its validation score measures agreement with the teachers, not accuracy.

In [10]:
# If the kernel was restarted meanwhile, labels are loaded from disk (no recomputation).
if "labels_a" not in globals():
    labels_a = bl.load_labels("labels_A")
if HRAST_AVAILABLE and "labels_b" not in globals():
    labels_b = bl.load_labels("labels_B")

# Neutral band for (B): almost half of the reviews are mixed and argmax labels them
# positive. The band rule treats balanced reviews as neutral
# (config.review_neutral_band). Sensitivity table first:
if HRAST_AVAILABLE:
    band_table = bl.neutral_band_table(labels_b, "B")
    display(band_table)
    ev.save_table(band_table, "02_B_neutral_band_sensitivity")
    labels_b = bl.apply_neutral_band(labels_b, "B")
    bl.save_labels(labels_b, "labels_B")

,negative,neutral,positive
neutral band,,,
±0.0,0.0990,0.0000,0.9010
±0.1,0.0805,0.0811,0.8384
±0.2,0.0628,0.1261,0.8110
±0.3,0.0522,0.1756,0.7722
±0.4,0.0366,0.2666,0.6968
±0.5,0.0278,0.3286,0.6436


17:48:21 | INFO    | src.evaluation | Table saved: 02_B_neutral_band_sensitivity.csv
17:48:21 | INFO    | src.bert_labelling | (B) neutral band ±0.25 -> {'positive': 0.788, 'neutral': 0.155, 'negative': 0.057}
17:48:21 | INFO    | src.bert_labelling | Labels saved: C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\data\labels\labels_B.parquet (203328 rows)


In [11]:
# Teachers for (C). The "neutral" class of (A) means "no opinion" (tweets), so if (B)
# is available it is the only teacher; otherwise the agreement condition would remove
# almost the whole neutral class.
TEACHERS = ("B",) if HRAST_AVAILABLE else ("A",)
teachers = labels_a.join(labels_b, how="inner") if HRAST_AVAILABLE else labels_a
pseudo = bl.select_pseudo_labels(teachers, teachers=TEACHERS)
display(pseudo["pseudo_label"].value_counts().to_frame("training examples"))

# An existing model (C) is reused: the pseudo-labels per class are capped (CFG), so the training
# set would not grow on the full data either, and (C) is only a comparison approach
# (the final label is (B)). RETRAIN_C = True retrains it.
RETRAIN_C = False
model_c_ready = (PATHS.models / "approach_c_self_training" / "config.json").exists()
if RETRAIN_C or not model_c_ready:
    result_c = bt.train_approach_c(working, pseudo)
    print("Validation (agreement with the teachers):", {k: round(v, 4) for k, v in result_c["val_metrics"].items()
                                                        if k in ("accuracy", "f1_macro")})
else:
    print("(C) already exists — reused, not retrained (it is applied to the current working set below).")

17:48:21 | INFO    | src.bert_labelling | Pseudo-label teachers: ['B']
17:48:21 | INFO    | src.bert_labelling | Pseudo-labels: 109146 of 203328 reviews (53.7%) | {'negative': 3508, 'neutral': 13180, 'positive': 92458}
17:48:21 | INFO    | src.bert_labelling | After capping at 15000 per class: {'positive': 15000, 'neutral': 13180, 'negative': 3508}


,training examples
pseudo_label,
positive,15000
neutral,13180
negative,3508


(C) already exists — reused, not retrained (it is applied to the current working set below).


In [12]:
scorer_c = bl.TransformerScorer(str(PATHS.models / "approach_c_self_training"))
labels_c = bl.label_approach_c(working, scorer=scorer_c)
bl.save_labels(labels_c, "labels_C")
display(labels_c["C_label"].value_counts(normalize=True).round(3).to_frame("share"))
print(f"Reviews (C) was trained on (excluded from comparisons): {labels_c['C_in_training'].mean():.1%}")

17:48:21 | INFO    | src.bert_labelling | Loading model: C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\results\models\approach_c_self_training (device: cuda, fp16: True)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

17:48:22 | INFO    | src.bert_labelling | Model type: 3class | labels: ['negative', 'neutral', 'positive']
17:48:22 | WARNING | src.bert_labelling | Cache 'C_review' is stale (input or model changed) — recomputing.
17:48:22 | INFO    | src.bert_labelling | [C_review] chunk 1/21 (10000 texts)
17:48:25 | INFO    | src.bert_labelling |   C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\results\models\approach_c_self_training: 50/157 batches (1269 texts/s)
17:48:29 | INFO    | src.bert_labelling |   C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\results\models\approach_c_self_training: 100/157 batches (948 texts/s)
17:48:37 | INFO    | src.bert_labelling |   C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\results\models\approach_c_self_training: 150/157 batches (651 texts/s)
17:48:39 | INFO    | src.bert_labelling | [C_review] chunk 2/21 (10000 texts)
17:48:42 | INFO    | src.bert_labelling |   C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THES

,share
C_label,
positive,0.630
neutral,0.349
negative,0.022


Reviews (C) was trained on (excluded from comparisons): 4.8%


## 5. Booking field test

Text from the `review_positive` field is expected to be positive and text from `review_negative` negative, since the guest chose the field. This is an independent check of all three models without labels.

In [13]:
field_set = lv.build_field_test_set(working)
field_scorers = {"A": scorers_a["3class"], "C": scorer_c}
if HRAST_AVAILABLE:
    field_scorers["B"] = scorer_b

field_preds = field_set.copy()
reports = []
for name, scorer in field_scorers.items():
    probs = scorer.predict_proba(field_set["text_bert"].tolist())
    field_preds[f"{name}_label"] = np.array(bl.LABELS)[probs.argmax(axis=1)]
    reports.append(lv.field_test_report(field_set["expected"], field_preds[f"{name}_label"]).assign(approach=name))

field_report = pd.concat(reports).reset_index().set_index(["approach", "field"])
display(field_report)
ev.save_table(field_report, "02_field_test")
dl.write_frame(field_preds, PATHS.labels / "field_test_predictions")

17:54:23 | INFO    | src.bert_labelling |   twitter-roberta-base-sentiment-latest: 50/63 batches (1164 texts/s)
17:54:28 | INFO    | src.bert_labelling |   C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\results\models\approach_c_self_training: 50/63 batches (1242 texts/s)
17:54:33 | INFO    | src.bert_labelling |   C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\results\models\approach_b_hrast: 50/63 batches (1263 texts/s)


n     hit  neutral  opposite
approach field                                           
A        review_negative  2000  0.4995   0.3520    0.1485
         review_positive  2000  0.9605   0.0325    0.0070
C        review_negative  2000  0.6645   0.2460    0.0895
         review_positive  2000  0.9420   0.0490    0.0090
B        review_negative  2000  0.7745   0.0990    0.1265
         review_positive  2000  0.9790   0.0050    0.0160

17:54:35 | INFO    | src.evaluation | Table saved: 02_field_test.csv


WindowsPath('C:/Users/seszt/Documents/BCE/7_felev/Szakdolgozat/THESIS/CODE/data/labels/field_test_predictions.parquet')

## 6. Final label

The final label comes from the approaches listed in `CFG.final_label_approaches` (ties: `CFG.tie_breaker_approach`); the final 1–10 score is the mean of their continuous scores. The `unanimous` column marks the reviews where all sources agree.

In [14]:
labelled = pipeline.step_combine()
display(labelled["label_final"].value_counts(normalize=True).round(3).to_frame("share"))
print(f"Unanimous: {labelled['unanimous'].mean():.1%} of the reviews")
display(labelled[["text", "score", "label_final", "score10_final", "n_agree"]].head(8))
print("\nNext: 03_label_validation.ipynb (runs on a CPU, locally)")

17:54:35 | INFO    | src.bert_labelling | Combining approaches: ['B'] (tie-breaker: B)
17:55:53 | INFO    | src.bert_labelling | Labels saved: C:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\data\labels\labels_final.parquet (203328 rows)
17:55:54 | INFO    | src.pipeline | Final label distribution: {'positive': 0.788, 'neutral': 0.155, 'negative': 0.057}


,share
label_final,
positive,0.788
neutral,0.155
negative,0.057


Unanimous: 100.0% of the reviews


,text,score,label_final,score10_final,n_agree
0,"Hope is relatively close to where I live, so this might just become a ""hop on the bike and cruise there for a weeke....",9.0,neutral,5,1
1,Close to the station and within walking distance of the centre,8.0,positive,10,1
2,"Excellent service. Excellent customer service, great location and the view, good breakfast spread, easy access to ci...",8.0,positive,7,1
3,"A beautiful hotel and grounds, very clean. A lovely big room with beautiful furnishings and excellent views of the g...",8.0,neutral,6,1
4,"Amazing Stay :). Incredible staff, convenient location, and nice comfortable communal lounge area. Rooms were clean,...",9.0,positive,8,1
5,"Great location, clean and very helpful staff. The location was great. Peaceful hotel and very clean. Staff were love...",7.0,positive,8,1
6,Everything about it was great. Fantastic location for old and new town. Hotel excellent and lovely. Breakfast was fa...,10.0,positive,8,1
7,"Location, cleanliness, breakfast Room rather small for price",8.0,negative,1,1



Next: 03_label_validation.ipynb (runs on a CPU, locally)
